# Stage 05 — Data Quality Monitoring

## Enterprise Data Quality & Governance Project

### Objective

The purpose of this stage is to introduce a simple monitoring layer on top of the Data Quality validation framework.

Instead of evaluating data quality only once, monitoring allows each validation run to be recorded and compared with previous runs.

The monitoring process captures:
- Run ID
- Run date
- Total records
- Number of Data Quality rules
- Total violations
- Valid records
- Quality rate
- Overall status
- Previous quality rate
- Change in quality rate


## 1. Load the Dataset

The same Customer 360 dataset is used for the monitoring run.


In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv("customers.csv")

# Apply the standardization established in earlier stages
df["income_level"] = df["income_level"].replace({
    "LOW": "Low"
})

df["location"] = df["location"].replace({
    "Chennal": "Chennai"
})

df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

df.head()


## 2. Define the Monitoring Rules

The monitoring run uses the same 15 Data Quality rules established in the validation framework.


In [ ]:
rules_catalog = [
    {"rule_id": "DQ001", "dimension": "Completeness", "field": "customer_id"},
    {"rule_id": "DQ002", "dimension": "Completeness", "field": "age"},
    {"rule_id": "DQ003", "dimension": "Completeness", "field": "gender"},
    {"rule_id": "DQ004", "dimension": "Completeness", "field": "location"},
    {"rule_id": "DQ005", "dimension": "Completeness", "field": "income_level"},
    {"rule_id": "DQ006", "dimension": "Completeness", "field": "signup_date"},
    {"rule_id": "DQ007", "dimension": "Uniqueness", "field": "customer_id"},
    {"rule_id": "DQ008", "dimension": "Validity", "field": "age"},
    {"rule_id": "DQ009", "dimension": "Validity", "field": "gender"},
    {"rule_id": "DQ010", "dimension": "Validity", "field": "income_level"},
    {"rule_id": "DQ011", "dimension": "Validity", "field": "location"},
    {"rule_id": "DQ012", "dimension": "Validity", "field": "signup_date"},
    {"rule_id": "DQ013", "dimension": "Consistency", "field": "income_level"},
    {"rule_id": "DQ014", "dimension": "Consistency", "field": "location"},
    {"rule_id": "DQ015", "dimension": "Consistency", "field": "signup_date"}
]

len(rules_catalog)


## 3. Run Data Quality Validation

The monitoring run reuses the validation logic from Stage 04.


In [ ]:
allowed_gender = ["Male", "Female"]
allowed_income = ["High", "Medium", "Low"]
allowed_locations = [
    "Mumbai",
    "Bangalore",
    "Chennai",
    "Hyderabad",
    "Kolkata",
    "Delhi"
]


def validate_rule(df, rule_id):
    field = next(rule["field"] for rule in rules_catalog if rule["rule_id"] == rule_id)

    if rule_id in ["DQ001", "DQ002", "DQ003", "DQ004", "DQ005", "DQ006"]:
        violations = df[field].isnull().sum()

    elif rule_id == "DQ007":
        violations = df[field].duplicated().sum()

    elif rule_id == "DQ008":
        violations = ((df[field] < 18) | (df[field] > 69)).sum()

    elif rule_id == "DQ009":
        violations = (~df[field].isin(allowed_gender)).sum()

    elif rule_id == "DQ010":
        violations = (~df[field].isin(allowed_income)).sum()

    elif rule_id == "DQ011":
        violations = (~df[field].isin(allowed_locations)).sum()

    elif rule_id == "DQ012":
        violations = df[field].isnull().sum()

    elif rule_id == "DQ013":
        violations = (~df[field].isin(allowed_income)).sum()

    elif rule_id == "DQ014":
        violations = (~df[field].isin(allowed_locations)).sum()

    elif rule_id == "DQ015":
        violations = df[field].isnull().sum()

    return violations


quality_results = []

for rule in rules_catalog:
    violations = validate_rule(df, rule["rule_id"])

    quality_results.append({
        "rule_id": rule["rule_id"],
        "dimension": rule["dimension"],
        "field": rule["field"],
        "violations": violations
    })

quality_results = pd.DataFrame(quality_results)

quality_results


## 4. Calculate the Current Run Summary

The validation results are summarized into one monitoring record.


In [ ]:
total_records = len(df)
total_rules = len(rules_catalog)
total_violations = quality_results["violations"].sum()
valid_records = total_records - total_violations

quality_rate = (
    (total_records * total_rules - total_violations)
    / (total_records * total_rules)
) * 100

status = "PASS" if total_violations == 0 else "FAIL"

quality_rate, status


## 5. Create the Monitoring Log

Each execution is stored as a monitoring run.

The first run is recorded as `RUN_001`.


In [ ]:
monitoring_log = []

run_01 = {
    "run_id": "RUN_001",
    "run_date": datetime.now().strftime("%Y-%m-%d"),
    "total_records": total_records,
    "total_rules": total_rules,
    "total_violations": total_violations,
    "valid_records": valid_records,
    "quality_rate": quality_rate,
    "status": status,
    "previous_quality_rate": None,
    "quality_rate_change": None
}

monitoring_log.append(run_01)

monitoring_df = pd.DataFrame(monitoring_log)

monitoring_df


## 6. Compare Quality Across Runs

Monitoring becomes more useful when a new run can be compared with the previous run.

The following example shows how a future run can calculate the change in quality rate.


In [ ]:
def create_monitoring_run(
    run_id,
    total_records,
    total_rules,
    total_violations,
    previous_quality_rate=None
):
    valid_records = total_records - total_violations

    quality_rate = (
        (total_records * total_rules - total_violations)
        / (total_records * total_rules)
    ) * 100

    status = "PASS" if total_violations == 0 else "FAIL"

    if previous_quality_rate is None:
        quality_rate_change = None
    else:
        quality_rate_change = quality_rate - previous_quality_rate

    return {
        "run_id": run_id,
        "run_date": datetime.now().strftime("%Y-%m-%d"),
        "total_records": total_records,
        "total_rules": total_rules,
        "total_violations": total_violations,
        "valid_records": valid_records,
        "quality_rate": quality_rate,
        "status": status,
        "previous_quality_rate": previous_quality_rate,
        "quality_rate_change": quality_rate_change
    }


## 7. Example of a Second Monitoring Run

This example demonstrates how the framework would record a later run using the previous quality rate.

It does not introduce a new data-quality problem; it only demonstrates the monitoring mechanism.


In [ ]:
previous_quality_rate = monitoring_df.iloc[-1]["quality_rate"]

run_02 = create_monitoring_run(
    run_id="RUN_002",
    total_records=total_records,
    total_rules=total_rules,
    total_violations=total_violations,
    previous_quality_rate=previous_quality_rate
)

run_02


## 8. Append the New Run to the Monitoring History

In [ ]:
monitoring_log.append(run_02)

monitoring_df = pd.DataFrame(monitoring_log)

monitoring_df


## 9. Monitoring Interpretation

For the current project data:

- `RUN_001` represents the first recorded monitoring run.
- The run contains **100,000 records**.
- **15 Data Quality rules** are evaluated.
- The current validated state has **0 violations**.
- The quality rate is **100%**.
- The status is **PASS**.

The second run demonstrates how future executions can be compared against a previous quality rate.


## 10. Why Monitoring Matters

A Data Quality framework becomes more useful when quality can be observed over time.

For example, future monitoring could detect:
- A decrease in quality rate
- New violations
- Repeated violations
- Changes in record volume
- A transition from PASS to FAIL

This changes the project from a one-time assessment into a basic **Data Quality monitoring process**.


# Stage 05 Conclusion

Stage 05 introduced a monitoring layer that records Data Quality validation runs and supports comparison between runs.

The current monitoring result is:

- **Run ID:** RUN_001
- **Records:** 100,000
- **Rules:** 15
- **Violations:** 0
- **Quality Rate:** 100%
- **Status:** PASS

The monitoring structure can be extended in future iterations with persistent storage, dashboards, alerts, scheduled execution, and additional historical runs.

The main outcome of this stage is the shift from **one-time Data Quality validation** toward **repeatable Data Quality monitoring**.
